In [ ]:
# Practice Problems Day 16

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's ideas: attention by hand, a self-attention layer, and reading the attention weights. Plan on about 45 minutes. Every cell runs in a few seconds on a laptop.

**Solutions** are in `HW-day16-solutions.ipynb` in this folder. Try every problem yourself first, then check your work.

**Reading:** Geron chapter 14, **Attention Mechanisms**.

GOALS:

1. Compute attention for one word: scores, softmax, weighted average.
2. Train a review classifier with self-attention and read its attention weights.
3. Explain queries, keys and values in your own words.

**How much of this should you write yourself?** Problem 1 gives you most of the code, you write the missing lines. Problem 2 lists the steps, and you can copy the classes from the notes. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 16 practice"
git push
```

---

In [ ]:
# RUN THIS. Imports, the review maker and the training function from the notes.
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words
vocab_size = len(vocab)

def make_reviews(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling
    y = (feeling < 5).long()
    return X, y

def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))
    return valid_accuracy

def show_weights(weights, words, title):
    plt.figure(figsize=(6, 5))
    plt.imshow(weights, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(words)), words, rotation=90)
    plt.yticks(range(len(words)), words)
    plt.xlabel("looks at (key)")
    plt.ylabel("word that is looking (query)")
    plt.colorbar()
    plt.title(title)
    plt.show()

## Problem 1: Attention for one word

**What to copy, what to write:** The numbers are given. Write the three lines the comments ask for. The notes do the same steps for "the pizza was cold".

The sentence is **"the soup tasted salty"**. Every key and query has 4 numbers, and every value has 2 numbers ("how much is this word about food" and "how much is it about taste").

**1a.** Fill in the cell to find the attention weights and the new vector for the word "salty".

**1b.** Which word gets the most weight? Does that make sense for the word "salty"? What does the new vector for "salty" say that the old value [0, 1] did not?

**1c.** Run it again **without** dividing by $\sqrt{4}$. What happens to the weights? Why is that a problem in a big model?

In [ ]:
# The keys, values and the query for "salty"
words = ["the", "soup", "tasted", "salty"]
K = torch.tensor([[0., 0, 0, 0],      # key for "the"
                  [0, 2, 1, 1],       # key for "soup"
                  [1, 0, 0, 1],       # key for "tasted"
                  [0, 0, 1, 0]])      # key for "salty"
V = torch.tensor([[0., 0],            # value for "the"
                  [1, 0],             # value for "soup"
                  [0, 0],             # value for "tasted"
                  [0, 1]])            # value for "salty"
q_salty = torch.tensor([0., 2, 0, 2])

# Step 1: the dot product of q_salty with every key, then divide by the square root of 4
# Write the scores line here

print("scaled scores:", scores)

# Step 2: softmax, so the weights add up to 1 (one dimension, so dim=0)
# Write the weights line here

print("weights:", weights)

# Step 3: the weighted average of the values
# Write the new_salty line here

print("new vector for salty:", new_salty)

In [ ]:
# 1c: the same thing without dividing by the square root of 4


*Your answers here.*

---

## Problem 2: Attention on 200 word reviews (steps outlined)

**What to copy, what to write:** Copy the `SelfAttention` and `ReviewAttention` classes from the notes (or better, type them out). Write the rest yourself.

**2a.** Make 200 word reviews (4,000 to train on with `seed=0`, 1,000 to validate on with `seed=1`) and train a `ReviewAttention` model on them for 3 epochs. How does it do? The plain RNN was no better than a coin flip on reviews half this long!

**2b.** Make one review with 12 words (`make_reviews(1, n_words=12, seed=3)`) and draw the trained model's attention weights with `show_weights`. Find the feeling word. How many rows put almost all their weight on it? The summary of the review is the **average** of the 12 new vectors. If every word got an equal share, the feeling word would get 1/12 = 0.083 of it. About how much does it get here?

In [ ]:
# Step 1: copy (or type) the SelfAttention class from the notes

# Step 2: copy (or type) the ReviewAttention class from the notes

# Step 3: make the 200 word reviews with make_reviews (train with seed=0, validate with seed=1)

# Step 4: torch.manual_seed(42), make a ReviewAttention model, train it for 3 epochs
# It prints 3 lines, one validation accuracy per epoch.


In [ ]:
# Step 1: make one 12 word review with make_reviews(1, n_words=12, seed=3) and print its words

# Step 2: inside torch.no_grad(), send it through the model's embed and attention layers to get the weights

# Step 3: show_weights(weights[0], your_list_of_words, "a title")
# You should see a 12 by 12 heat map.


*Your answers here.*

---

## Problem 3: In words

**3a.** Explain queries, keys and values to a friend in DATA 101. Use a short sentence of your own (not the pizza or the soup) and say what one word's query would be looking for. Three or four sentences, no code.

**3b.** Your friend says: "So attention just takes the average of the words. We could do that without all the queries and keys." What is different about attention? (Hint: in the pizza example, what were the weights for the "cold" row, and for the "the" row?)

*Your answers here.*

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Check your Problem 1 answer with the built in version: `torch.nn.functional.scaled_dot_product_attention`. It wants matrices, so use `q_salty.reshape(1, 4)` for the query.

**B.** Make a 1,000 word review set (2,000 to train on) and time one epoch of `ReviewAttention` with `import time` and `time.time()`. How does it compare to the 200 word reviews?

In [ ]:
# your code here

*Your answers here.*